#### 11 — Compare Custom CNN vs Fine-Tuned ResNet18

##### Objective

Evaluate and compare the two trained image-classification models on the
same untouched test dataset:

1. Custom CNN
2. Fine-tuned ResNet18

Both models are loaded independently from MLflow.

The notebook compares:

- test accuracy
- per-class precision
- per-class recall
- per-class F1 score
- confusion matrices
- missed defects
- false defect alarms
- model agreement and disagreement

The test dataset is used only for final evaluation.
It is not used for training, fine-tuning, checkpoint selection,
or hyperparameter tuning.

##### Important preprocessing rule

Each model must receive the preprocessing used for that model:

- Custom CNN → custom CNN evaluation preprocessing
- ResNet18 → pretrained ResNet18 evaluation preprocessing

The underlying test images and class labels remain the same.

##### 🔍 Explainable AI (XAI)

This notebook identifies where the models agree and disagree.

It tells us **what** each model predicted, but not yet **why**.
Later Grad-CAM analysis will investigate why selected predictions occurred.

##### ⚖️ Responsible AI (RAI)

Overall accuracy is not sufficient for a quality-inspection system.

We will pay particular attention to:

- missed defects
- false defect alarms
- per-class recall
- failure cases

##### 2. Imports

In [0]:
import mlflow
import mlflow.pytorch

import pandas as pd
import torch

from sklearn.metrics import confusion_matrix

from src.data_preparation import create_image_dataloaders
from src.model_evaluation import evaluate_classifier

##### 3. Define the persisted models

In [0]:
CUSTOM_CNN_RUN_ID =  "8b154a58aaae400fbb941a7d8299dc19"

RESNET18_RUN_ID =  "3f9b44ae55a449c5850f0b48c4ee4fa6"

CUSTOM_CNN_MODEL_URI =  f"runs:/{CUSTOM_CNN_RUN_ID}/model"

RESNET18_MODEL_URI =  f"runs:/{RESNET18_RUN_ID}/model"

print(
    "Custom CNN:",
    CUSTOM_CNN_MODEL_URI,
)

print(
    "Fine-tuned ResNet18:",
    RESNET18_MODEL_URI,
)

Notice what we're identifying:

``` text

Custom CNN
    ↓
best CNN persisted from Notebook 06

ResNet18
    ↓
selected fine-tuned Stage-2 model
persisted from Notebook 10

```

We are not training anything here.

##### 4. Select device

In [0]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Evaluation device:", device)

##### 5. Recreate both data pipelines

In [0]:
cnn_data = create_image_dataloaders(preprocessing_mode="custom_cnn",)

resnet_data = create_image_dataloaders(preprocessing_mode="pretrained",)

##### 6. Verify dataset consistency

In [0]:
#Before loading the models, let's prove we're comparing the same test dataset.

print(
    "CNN test images:",
    len(cnn_data.test_dataset),
)

print(
    "ResNet test images:",
    len(resnet_data.test_dataset),
)

print(
    "CNN classes:",
    cnn_data.class_names,
)

print(
    "ResNet classes:",
    resnet_data.class_names,
)

print(
    "CNN class mapping:",
    cnn_data.class_to_idx,
)

print(
    "ResNet class mapping:",
    resnet_data.class_to_idx,
)

In [0]:
assert (
    len(cnn_data.test_dataset)
    == len(resnet_data.test_dataset)
)

assert (
    cnn_data.class_names
    == resnet_data.class_names
)

assert (
    cnn_data.class_to_idx
    == resnet_data.class_to_idx
)

print(
    "Dataset size and class mapping checks passed."
)

In [0]:
cnn_test_samples = (
    cnn_data.test_dataset.samples
)

resnet_test_samples = (
    resnet_data.test_dataset.samples
)

cnn_test_paths = [
    path
    for path, _ in cnn_test_samples
]

resnet_test_paths = [
    path
    for path, _ in resnet_test_samples
]

cnn_test_labels = [
    label
    for _, label in cnn_test_samples
]

resnet_test_labels = [
    label
    for _, label in resnet_test_samples
]

assert (
    cnn_test_paths
    == resnet_test_paths
)

assert (
    cnn_test_labels
    == resnet_test_labels
)

print(
    "Exact test image paths and labels match."
)

##### 7. Load both persisted MLflow models

In [0]:
cnn_model = mlflow.pytorch.load_model(CUSTOM_CNN_MODEL_URI,)

resnet_model = mlflow.pytorch.load_model(RESNET18_MODEL_URI,)

cnn_model = cnn_model.to(device)
resnet_model = resnet_model.to(device)

cnn_model.eval()
resnet_model.eval()

print(
    "Custom CNN loaded successfully."
)

print(
    "Fine-tuned ResNet18 loaded successfully."
)

##### 8. Evaluate Custom CNN

In [0]:
#Now we'll reuse the production evaluation function we already created.

cnn_result = evaluate_classifier(
    model=cnn_model,
    data_loader=cnn_data.test_loader,
    device=device,
    num_classes=cnn_data.num_classes,
)

print(
    "Custom CNN Test Accuracy:",
    f"{cnn_result.accuracy:.4f}",
)

In [0]:
import inspect

print(
    inspect.signature(
        evaluate_classifier
    )
)

In [0]:
help(evaluate_classifier)

##### 9. Evaluate fine-tuned ResNet18

In [0]:
resnet_result = evaluate_classifier(
    model=resnet_model,
    data_loader=resnet_data.test_loader,
    device=device,
    num_classes=resnet_data.num_classes,
)


In [0]:
print(
    "Fine-Tuned ResNet18 Test Accuracy:",
    f"{resnet_result.accuracy:.4f}",
)

##### 10. Inspect the returned EvaluationResult

In [0]:
print("Custom CNN Evaluation Result")
print(cnn_result)

print()

print("Fine-Tuned ResNet18 Evaluation Result")
print(resnet_result)

In [0]:
print(
    "EvaluationResult fields:",
    cnn_result.__dict__.keys(),
)

##### 11. Confirm the confusion matrices

If confusion_matrix is one of the returned fields, we'll use that directly. If not, we already imported confusion_matrix, and can calculate it from the returned labels/predictions.

The expected interpretation will be:

                    PREDICTED
                defect       OK
ACTUAL defect      ?          ?
ACTUAL OK          ?          ?

For the CNN we already know:

                    PREDICTED
                defect       OK
ACTUAL defect     385         68
ACTUAL OK          52        210

So:

- 68 = missed defects
- 52 = false defect alarms

Now I'm particularly interested in the ResNet18 matrix because only about 3 total errors appear likely from the accuracy.

Imagine the difference conceptually:

``` text

Custom CNN
715 images
   ↓
120 errors

ResNet18
715 images
   ↓
~3 errors

```

We'll verify those exact counts rather than assuming them.

Why this is such a useful transfer-learning result

Remember where the two models started:

``` text

CUSTOM CNN
Random weights
     ↓
Only 5,638 training images
     ↓
Learns visual representation from scratch
     ↓
83.22% test accuracy

versus:

RESNET18
Pretrained visual representation
     ↓
Our new 512 → 2 classifier
     ↓
Feature extraction
     ↓
Fine-tune layer4
     ↓
99.58% test accuracy

```

This gives you a concrete example of why transfer learning is so useful when the project dataset is relatively limited. The pretrained model didn't already know def_front versus ok_front; it brought reusable visual feature extraction, and we adapted that representation to our task.

⚖️ RAI — don't stop at 99.58%

The result is excellent, but now our questions actually become more interesting:

``` text

99.58% accuracy
      ↓
Which images were wrong?
      ↓
Were any actual defects classified as OK?
      ↓
How confident were those mistakes?
      ↓
Are train/validation/test images genuinely independent?
      ↓
Could near-duplicate images exist?
      ↓
Is the model relying on the casting surface
or some irrelevant visual cue?

```
The first questions belong to ⚖️ RAI model assessment.

The last question will lead directly into 🔍 XAI / Grad-CAM.

And checking for duplicates/near-duplicates is particularly worthwhile now. A very high score does not itself mean leakage, but at this level of performance we should verify dataset independence before making strong claims about generalization.

In [0]:
print("Custom CNN Evaluation Result")
print(cnn_result)

print()

print("Fine-Tuned ResNet18 Evaluation Result")
print(resnet_result)

print()

print(
    "EvaluationResult fields:",
    cnn_result.__dict__.keys(),
)

##### 12. Build the side-by-side comparison

In [0]:
class_names = cnn_data.class_names

comparison_df = pd.DataFrame(
    {
        "Metric": [
            "Test Accuracy",
            "Defect Precision",
            "Defect Recall",
            "Defect F1",
            "OK Precision",
            "OK Recall",
            "OK F1",
            "Missed Defects",
            "False Defect Alarms",
        ],
        "Custom CNN": [
            cnn_result.accuracy,
            cnn_result.precision[0],
            cnn_result.recall[0],
            cnn_result.f1[0],
            cnn_result.precision[1],
            cnn_result.recall[1],
            cnn_result.f1[1],
            cnn_result.confusion_matrix[0, 1],
            cnn_result.confusion_matrix[1, 0],
        ],
        "Fine-Tuned ResNet18": [
            resnet_result.accuracy,
            resnet_result.precision[0],
            resnet_result.recall[0],
            resnet_result.f1[0],
            resnet_result.precision[1],
            resnet_result.recall[1],
            resnet_result.f1[1],
            resnet_result.confusion_matrix[0, 1],
            resnet_result.confusion_matrix[1, 0],
        ],
    }
)

display(comparison_df)

##### 13. Compare the confusion matrices

In [0]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

In [0]:
cnn_display = ConfusionMatrixDisplay(
    confusion_matrix=cnn_result.confusion_matrix,
    display_labels=class_names,
)

cnn_display.plot()

plt.title(
    "Custom CNN — Test Confusion Matrix"
)

plt.show()

In [0]:
resnet_display = ConfusionMatrixDisplay(
    confusion_matrix=resnet_result.confusion_matrix,
    display_labels=class_names,
)

resnet_display.plot()

plt.title(
    "Fine-Tuned ResNet18 — Test Confusion Matrix"
)

plt.show()

So ResNet18 reduced the total number of errors from: 120 → 3

That's a reduction of: 117 fewer errors

More importantly for defect inspection:

Missed defects

Custom CNN:       68
Fine-tuned ResNet: 3

Reduction:        65

##### 14. Model agreement/disagreement

In [0]:
labels = cnn_result.labels

cnn_predictions = (
    cnn_result.predictions
)

resnet_predictions = (
    resnet_result.predictions
)

both_correct = (
    (cnn_predictions == labels)
    & (resnet_predictions == labels)
)

cnn_wrong_resnet_correct = (
    (cnn_predictions != labels)
    & (resnet_predictions == labels)
)

cnn_correct_resnet_wrong = (
    (cnn_predictions == labels)
    & (resnet_predictions != labels)
)

both_wrong = (
    (cnn_predictions != labels)
    & (resnet_predictions != labels)
)

print(
    "Both correct:",
    both_correct.sum(),
)

print(
    "CNN wrong / ResNet correct:",
    cnn_wrong_resnet_correct.sum(),
)

print(
    "CNN correct / ResNet wrong:",
    cnn_correct_resnet_wrong.sum(),
)

print(
    "Both wrong:",
    both_wrong.sum(),
)

##### 15. Create an image-level comparison table

In [0]:
prediction_comparison_df = pd.DataFrame(
    {
        "image_path": cnn_test_paths,
        "actual_label": labels,
        "cnn_prediction": cnn_predictions,
        "resnet_prediction": resnet_predictions,
        "cnn_confidence": (
            cnn_result.probabilities.max(
                axis=1
            )
        ),
        "resnet_confidence": (
            resnet_result.probabilities.max(
                axis=1
            )
        ),
    }
)

In [0]:
label_lookup = {
    index: name
    for index, name
    in enumerate(class_names)
}

prediction_comparison_df[
    "actual_class"
] = prediction_comparison_df[
    "actual_label"
].map(label_lookup)

prediction_comparison_df[
    "cnn_predicted_class"
] = prediction_comparison_df[
    "cnn_prediction"
].map(label_lookup)

prediction_comparison_df[
    "resnet_predicted_class"
] = prediction_comparison_df[
    "resnet_prediction"
].map(label_lookup)

In [0]:
prediction_comparison_df[
    "cnn_correct"
] = (
    prediction_comparison_df[
        "actual_label"
    ]
    == prediction_comparison_df[
        "cnn_prediction"
    ]
)

prediction_comparison_df[
    "resnet_correct"
] = (
    prediction_comparison_df[
        "actual_label"
    ]
    == prediction_comparison_df[
        "resnet_prediction"
    ]
)

##### 16. Find the three ResNet18 mistakes

In [0]:
resnet_errors_df = (
    prediction_comparison_df[
        ~prediction_comparison_df[
            "resnet_correct"
        ]
    ]
    .copy()
)

display(
    resnet_errors_df[
        [
            "image_path",
            "actual_class",
            "cnn_predicted_class",
            "cnn_confidence",
            "resnet_predicted_class",
            "resnet_confidence",
        ]
    ]
)

In [0]:
assert len(resnet_errors_df) == 3

##### 19. Build the exact-duplicate check

In [0]:
import hashlib
from pathlib import Path

In [0]:
def calculate_file_hash(
    file_path: str,
) -> str:
    """
    Calculate the SHA-256 hash of a file.
    """

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        while chunk := file.read(8192):
            sha256.update(chunk)

    return sha256.hexdigest()

##### 20. Get training and test image paths

For this check we want the original source folders, not augmented tensors.

Since the dataset structure is:
``` text

casting_data/
├── train/
│   ├── def_front/
│   └── ok_front/
└── test/
    ├── def_front/
    └── ok_front/

```

use the configured raw-data path:

In [0]:
from src.project_config import RAW_DATA_DIR
from pathlib import Path

In [0]:
train_directory = Path(
    RAW_DATA_DIR
) / "train"

test_directory = Path(
    RAW_DATA_DIR
) / "test"

image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
}

train_image_paths = sorted(
    path
    for path in train_directory.rglob("*")
    if path.suffix.lower()
    in image_extensions
)

test_image_paths = sorted(
    path
    for path in test_directory.rglob("*")
    if path.suffix.lower()
    in image_extensions
)

print(
    "Original training images:",
    len(train_image_paths),
)

print(
    "Original test images:",
    len(test_image_paths),
)

##### 21. Calculate hashes

In [0]:
import hashlib

def calculate_file_hash(
    file_path: str,
) -> str:
    """
    Calculate the SHA-256 hash of a file.
    """

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        while chunk := file.read(8192):
            sha256.update(chunk)

    return sha256.hexdigest()


In [0]:
train_hash_to_paths = {}


for path in train_image_paths:
    file_hash = calculate_file_hash(
        str(path)
    )

    train_hash_to_paths.setdefault(
        file_hash,
        [],
    ).append(
        str(path)
    )

In [0]:
test_hash_to_paths = {}

for path in test_image_paths:
    file_hash = calculate_file_hash(
        str(path)
    )

    test_hash_to_paths.setdefault(
        file_hash,
        [],
    ).append(
        str(path)
    )

In [0]:
#Now find hashes appearing in both:

shared_hashes = (
    set(train_hash_to_paths)
    & set(test_hash_to_paths)
)

print(
    "Exact train/test duplicate hashes:",
    len(shared_hashes),
)

##### 22. Inspect duplicates if any exist

In [0]:
import pandas as pd

duplicate_records = []

for file_hash in shared_hashes:
    for train_path in train_hash_to_paths[
        file_hash
    ]:
        for test_path in test_hash_to_paths[
            file_hash
        ]:
            duplicate_records.append(
                {
                    "hash": file_hash,
                    "train_path": train_path,
                    "test_path": test_path,
                }
            )

duplicate_df = pd.DataFrame(
    duplicate_records
)

print(
    "Exact duplicate train/test pairs:",
    len(duplicate_df),
)

if not duplicate_df.empty:
    display(duplicate_df)

How we'll interpret it

If: Exact duplicate train/test pairs: 0

then we can say: No exact file duplicates were detected between the original training and test directories using SHA-256 hashing.

We should not say: There is definitely no leakage.

because this test establishes only:

Exact byte-identical duplicates
              ✓ checked

It does not detect:

- same physical casting photographed twice
- near-duplicate images
- slightly cropped duplicate
- brightness-modified duplicate
- sequential highly similar frames

Those require perceptual/semantic analysis.

That's an important ⚖️ RAI distinction:

No exact duplicates
        ≠
Proof of complete dataset independence

##### 23. Final comparison summary

In [0]:
cnn_errors = (
    cnn_result.predictions
    != cnn_result.labels
).sum()

resnet_errors = (
    resnet_result.predictions
    != resnet_result.labels
).sum()

cnn_missed_defects = (
    cnn_result.confusion_matrix[0, 1]
)

resnet_missed_defects = (
    resnet_result.confusion_matrix[0, 1]
)

print(
    "CUSTOM CNN"
)

print(
    f"Accuracy: "
    f"{cnn_result.accuracy:.2%}"
)

print(
    f"Total errors: "
    f"{cnn_errors}"
)

print(
    f"Missed defects: "
    f"{cnn_missed_defects}"
)

print()

print(
    "FINE-TUNED RESNET18"
)

print(
    f"Accuracy: "
    f"{resnet_result.accuracy:.2%}"
)

print(
    f"Total errors: "
    f"{resnet_errors}"
)

print(
    f"Missed defects: "
    f"{resnet_missed_defects}"
)

##### 24. evaluate the clean subset

In [0]:
#Let's add a column identifying contaminated test images.

test_hashes = [
    calculate_file_hash(path)
    for path in cnn_test_paths
]

prediction_comparison_df[
    "exact_development_duplicate"
] = [
    file_hash in shared_hashes
    for file_hash in test_hashes
]

In [0]:
print(
    "Total test images:",
    len(prediction_comparison_df),
)

print(
    "Exact train duplicates:",
    prediction_comparison_df[
        "exact_development_duplicate"
    ].sum(),
)

print(
    "Non-duplicate test images:",
    (
        ~prediction_comparison_df[
            "exact_development_duplicate"
        ]
    ).sum(),
)

##### 25. Evaluate the non-duplicate test subset

In [0]:
clean_test_mask = (
    ~prediction_comparison_df[
        "exact_development_duplicate"
    ].to_numpy()
)

print(
    "Non-duplicate test images:",
    clean_test_mask.sum(),
)

In [0]:
clean_labels = (
    cnn_result.labels[
        clean_test_mask
    ]
)

clean_cnn_predictions = (
    cnn_result.predictions[
        clean_test_mask
    ]
)

clean_resnet_predictions = (
    resnet_result.predictions[
        clean_test_mask
    ]
)

In [0]:
assert len(clean_labels) == 651

assert (
    len(clean_cnn_predictions)
    == len(clean_labels)
)

assert (
    len(clean_resnet_predictions)
    == len(clean_labels)
)

print(
    "Clean-subset arrays verified."
)

##### 26. Calculate clean-subset accuracy

In [0]:
clean_cnn_accuracy = (
    clean_cnn_predictions
    == clean_labels
).mean()

clean_resnet_accuracy = (
    clean_resnet_predictions
    == clean_labels
).mean()

print(
    "Custom CNN Clean Test Accuracy:",
    f"{clean_cnn_accuracy:.4f}",
)

print(
    "Fine-Tuned ResNet18 Clean Test Accuracy:",
    f"{clean_resnet_accuracy:.4f}",
)

##### 27. Clean-subset confusion matrices

In [0]:
#Accuracy still isn't enough, especially because defect recall matters.

clean_cnn_cm = confusion_matrix(
    clean_labels,
    clean_cnn_predictions,
    labels=[0, 1],
)

clean_resnet_cm = confusion_matrix(
    clean_labels,
    clean_resnet_predictions,
    labels=[0, 1],
)

print(
    "Custom CNN Clean Confusion Matrix:"
)

print(
    clean_cnn_cm
)

print()

print(
    "Fine-Tuned ResNet18 "
    "Clean Confusion Matrix:"
)

print(
    clean_resnet_cm
)

##### 28. Were ResNet's three mistakes duplicates?

In [0]:
resnet_error_mask = (
    resnet_result.predictions
    != resnet_result.labels
)

resnet_error_duplicate_flags = (
    prediction_comparison_df.loc[
        resnet_error_mask,
        "exact_development_duplicate",
    ]
)

print(
    "ResNet errors:",
    resnet_error_mask.sum(),
)

print(
    "ResNet errors that were "
    "exact development duplicates:",
    resnet_error_duplicate_flags.sum(),
)

print(
    "ResNet errors that were "
    "non-duplicates:",
    (
        ~resnet_error_duplicate_flags
    ).sum(),
)

3 ResNet mistakes
      │
      ├── duplicate images?
      │
      └── genuinely non-duplicate images?

``` text
Model comparison
      ↓
Very high result
      ↓
Question the result
      ↓
Audit dataset integrity
      ↓
Find 64 exact duplicates
      ↓
Quantify their effect
      ↓
Re-evaluate 651 non-duplicates
      ↓
Document limitation transparently
      ↓
Move to XAI failure analysis

```

##### Conclusion

Two independently persisted models were evaluated on the same original
715-image test split.

##### Custom CNN

- Test accuracy: 83.22%
- Total errors: 120
- Defect recall: 84.99%
- Missed defects: 68
- False defect alarms: 52

##### Fine-Tuned ResNet18

- Test accuracy: 99.58%
- Total errors: 3
- Defect recall: 99.34%
- Missed defects: 3
- False defect alarms: 0

The fine-tuned ResNet18 showed substantially stronger test performance
than the custom CNN across the evaluated classification metrics.

At the image level:

- both models correctly classified 595 images
- ResNet18 correctly classified 117 images that the custom CNN missed
- there were no images that the custom CNN classified correctly while
  ResNet18 classified them incorrectly
- both models misclassified the same 3 defect images

The three remaining ResNet18 errors were all false negatives:
actual defective products classified as OK.

One of these errors was predicted as OK with approximately 98.46%
softmax confidence, demonstrating that high model confidence does not
guarantee prediction correctness.

##### Transfer Learning

The custom CNN learned its visual representation from the project
training dataset starting from randomly initialized parameters.

ResNet18 started with pretrained visual features, learned a new
project-specific classifier, and then adapted its later feature
extraction layer through partial fine-tuning.

For this dataset, the transfer-learning approach produced substantially
stronger classification performance than the custom CNN.

##### ⚖️ Responsible AI (RAI)

Overall accuracy alone is not sufficient for a manufacturing quality
inspection system.

Missed defects are particularly important because a defective product
classified as OK could pass an automated inspection step.

The fine-tuned ResNet18 reduced missed defects from 68 to 3, but the
remaining false negatives still require investigation.

A SHA-256 data-integrity audit identified 64 exact byte-identical images
shared between the original 6,633-image development pool and the
715-image test split.

The original development pool was subsequently divided into training
and validation subsets, so these 64 images represent exact
development/test duplication. This means the original test split is not
fully independent of the model-development data.

To measure the effect of this issue, both models were also assessed on
the 651 test images that had no exact SHA-256 match in the development
pool.

##### Non-Duplicate Test Subset

- Number of images: 651
- Custom CNN accuracy: 83.26%
- Fine-tuned ResNet18 accuracy: 99.54%
- Custom CNN missed defects: 68
- Fine-tuned ResNet18 missed defects: 3
- Custom CNN false defect alarms: 41
- Fine-tuned ResNet18 false defect alarms: 0

Removing the 64 exact duplicates produced very little change in model
accuracy:

- Custom CNN: 83.22% → 83.26%
- Fine-tuned ResNet18: 99.58% → 99.54%

All three ResNet18 errors occurred among the non-duplicate test images.
None of the three errors was an exact development/test duplicate.

Therefore, exact development/test duplication is an important
evaluation-integrity limitation that must be documented, but the exact
duplicates do not appear to explain the fine-tuned ResNet18's very high
classification performance.

The SHA-256 audit detects exact byte-identical files only. It does not
rule out near-duplicate images, repeated photographs of the same
physical product, visually similar sequential samples, or other forms
of dataset similarity.

The very high validation and test performance should therefore still be
interpreted together with further data-quality and failure-case
assessment.

##### 🔍 Explainable AI (XAI)

Performance metrics tell us which predictions were correct or
incorrect, but they do not explain why a model made a prediction.

The three non-duplicate defect images misclassified by both models are
particularly valuable failure cases for later Grad-CAM analysis.

One of these images was classified as OK by the fine-tuned ResNet18
with approximately 98.46% softmax confidence, making it especially
useful for investigating the difference between model confidence,
prediction correctness, and model explanation.

The next phase will investigate which image regions influenced the
model's decisions.